In [1]:
import geopandas as gpd
import pandas as pd

gdf = gpd.read_file("export.geojson")

# order: amenity > tourism > leisure > shop
gdf['type'] = gdf['amenity'].combine_first(
                gdf['tourism'].combine_first(
                    gdf['leisure'].combine_first(
                        gdf['shop']
                    )
                )
            )

# count the number of each primary POI type
type_counts = gdf['type'].value_counts().reset_index()
type_counts.columns = ['POI Type', 'Count']

print("Number of records of each type: ")
print(type_counts)

print(f"\nTotal records: {len(gdf)}")

Number of records of each type: 
            POI Type  Count
0         restaurant   3306
1               cafe   1169
2                bar    623
3               park    487
4            gallery    223
5            toilets    163
6                atm    137
7         attraction     63
8   department_store     57
9             museum     54
10       arts_centre      3
11       picnic_site      1
12             clock      1
13     social_centre      1
14       planetarium      1
15       boat_rental      1

Total records: 6290


#### Delete irrelevant categories

In [2]:
gdf = gdf.cx[-74.03:-73.90, 40.68:40.88]

valid_types = [
    'restaurant', 'cafe', 'bar', 'park',
    'gallery', 'toilets', 'atm', 'attraction',
    'department_store', 'museum'
]

gdf_filtered = gdf[gdf['type'].isin(valid_types)]

print(f"Filtered total records: {len(gdf_filtered)}")
print(gdf_filtered['type'].value_counts())

Filtered total records: 6113
type
restaurant          3275
cafe                1156
bar                  614
park                 437
gallery              223
toilets              132
atm                  107
attraction            62
department_store      54
museum                53
Name: count, dtype: int64


In [3]:
# Remove records that have no name, except for toilets
gdf_filtered = gdf_filtered[
    (gdf_filtered['type'] == 'toilets') |
    (gdf_filtered['name'].notna() & (gdf_filtered['name'].str.strip() != ''))
]

print(f"\nRemaining records after name-type cleaning: {len(gdf_filtered)}")
print(gdf_filtered['type'].value_counts())

gdf_filtered.to_file("6000POI.geojson", driver="GeoJSON")


Remaining records after name-type cleaning: 5945
type
restaurant          3250
cafe                1145
bar                  612
park                 361
gallery              220
toilets              132
atm                   63
attraction            60
museum                52
department_store      50
Name: count, dtype: int64


In [4]:
import geopandas as gpd
import pandas as pd

# each type 50 records
samples = (
    gdf_filtered.groupby('type', group_keys=False)
                .apply(lambda x: x.sample(n=min(len(x), 50), random_state=42))
)

# Reset index and ensure GeoDataFrame retains original CRS
gdf_sampled = gpd.GeoDataFrame(samples.reset_index(drop=True), crs=gdf_filtered.crs)

# If total records exceed 500, randomly sample down to exactly 500
if len(gdf_sampled) > 500:
    gdf_sampled = gdf_sampled.sample(n=500, random_state=42)

print(f"\nFinal sampled record count: {len(gdf_sampled)}")
print(gdf_sampled['type'].value_counts())

# export
gdf_sampled.to_file("500POI.geojson", driver="GeoJSON")


Final sampled record count: 500
type
atm                 50
attraction          50
bar                 50
cafe                50
department_store    50
gallery             50
museum              50
park                50
restaurant          50
toilets             50
Name: count, dtype: int64


C:\Users\kexun\AppData\Local\Temp\ipykernel_13048\614978184.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 50), random_state=42))
